We replicate Jiang et al. (2023)

In [2]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

EXCEL_DIR = Path('./data_kospi ')
OUTPUT_DIR = Path('./processed_data')
TRAIN_VALID_START, TRAIN_VALID_END = '2016-07-01', '2024-06-30'
TEST_START, TEST_END = '2024-07-01', '2026-06-30'

SEED = 42
HORIZONS = (20, 60)  
WINDOW_STEP = 1
SPECS = {
    5: {'W': 15, 'H': 32, 'price_h': 25, 'vol_h': 6},
    20: {'W': 60, 'H': 64, 'price_h': 51, 'vol_h': 12},
    60: {'W': 180, 'H': 96, 'price_h': 76, 'vol_h': 19},
}


In [3]:
def load_history(directory):

    """convert Excel files to a single DataFrame with columns: Date, Ticker, Open, High, Low, Close, Volume, RET"""

    paths = [p for p in sorted(Path(directory).glob('*.xlsx')) if not p.name.startswith('~$')]
    if not paths:
        raise FileNotFoundError(f'Excel 파일이 없습니다: {directory}')
    aliases = {'date': 'Date', 'ticker': 'Ticker', 'open': 'Open',
               'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume',
               'ret': 'RET', '시가': 'Open', '고가': 'High', '저가': 'Low',
               '종가': 'Close', '거래량': 'Volume', '등락률': 'ReturnPct'}
    frames = []
    for path in paths:
        frame = pd.read_excel(path)
        frame = frame.rename(columns={c: aliases.get(str(c).strip().lower(), c) for c in frame})
        required = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
        missing = set(required) - set(frame)
        if missing:
            raise ValueError(f'{path}: 필수 컬럼 누락 {missing}')
        if 'RET' not in frame:
            if 'ReturnPct' not in frame:
                raise ValueError(f'{path}: RET(소수) 또는 등락률(%)이 필요합니다.')
            frame['RET'] = pd.to_numeric(frame['ReturnPct'], errors='coerce') / 100
        frame['Date'] = pd.to_datetime(frame['Date'].astype(str).str.replace(r'\.0$', '', regex=True))
        frame['Ticker'] = frame['Ticker'].astype('string').str.replace(r'\.0$', '', regex=True).str.zfill(6)
        if frame[['Date', 'Ticker']].isna().any().any():
            raise ValueError(f'{path}: Date/Ticker 결측')
        for col in required[2:] + ['RET']:
            frame[col] = pd.to_numeric(frame[col], errors='coerce')
        frames.append(frame[required + ['RET']])
    history = pd.concat(frames, ignore_index=True).sort_values(['Ticker', 'Date'])
    if history.duplicated(['Ticker', 'Date']).any():
        raise ValueError('동일 종목/날짜 중복 데이터가 있습니다.')
    return history


def reconstruct_prices(window):

    """ first close = 1, subsequent closes = previous reconstructed close * (1 + RET); OHLC maintains the same ratio for the day. """

    result = window.copy()
    returns = result['RET'].to_numpy(float)
    # since the first day RET is the return of the previous date, it is not used.
    if not np.isfinite(returns[1:]).all() or (returns[1:] <= -1).any():
        raise ValueError('가격 재구성에 필요한 수익률이 유효하지 않습니다.')
    close = np.r_[1.0, np.cumprod(1 + returns[1:])]
    if not np.isfinite(close).all() or (close <= 0).any():
        raise ValueError('재구성 가격이 유효하지 않습니다.')
    raw_close = result['Close'].where(result['Close'] > 0)
    for column in ['Open', 'High', 'Low']:
        result[column] = result[column] / raw_close * close
    result['Close'] = np.where(raw_close.notna(), close, np.nan)
    if not np.isfinite(result['Close'].iloc[0]):
        raise ValueError('첫날 종가가 유효하지 않습니다.')
    return result


def future_return(work, end, horizon):

    """ compute future return for the given horizon starting from the end index. """
    values = work['RET'].iloc[end + 1:end + horizon + 1].to_numpy(float)
    if len(values) != horizon or not np.isfinite(values).all() or (values < -1).any():
        return np.nan
    return float(np.prod(1 + values) - 1)


In [4]:
def generate_ohlcv_image(sub_df, sample_size):
    
    """
    replicate the OHLCV image generation process as described in the paper, following the specifications in Figure 3 & 4 and Section 2.2:
    - Grayscale 1-channel (Black: 0, White: 255)
    - The moving average (MA) is represented as a single pixel dot in the middle column (x_mid) for each date
    - Intermediate trading halts (NaN/0) are left as 0 (Blank)
    """
    
    sub_df = reconstruct_prices(sub_df).reset_index(drop=True)
    spec = SPECS[sample_size]
    width, height = spec['W'], spec['H']
    price_h, vol_h = spec['price_h'], spec['vol_h']

    img = np.zeros((height, width), dtype=np.uint8)

    # 해당 윈도우 기간 내의 이동평균선 계산 (Window = sample_size)
    ma_series = sub_df['Close'].rolling(window=sample_size).mean()

    # Price & Volume Min-Max Scaling (분모 0 방지)
    p_min = sub_df[['Open', 'High', 'Low', 'Close']].min().min()
    p_max = sub_df[['Open', 'High', 'Low', 'Close']].max().max()
    p_range = (p_max - p_min) if p_max != p_min else 1.0

    v_max = sub_df['Volume'].max()
    v_max = v_max if v_max > 0 else 1.0

    for i in range(sample_size):
        row = sub_df.iloc[i]

        # [논문 조건]: 중간 결측치/거래정지일은 픽셀을 칠하지 않고 0(Blank) 처리
        if not np.isfinite(row[['Open', 'High', 'Low', 'Close', 'Volume']].to_numpy(float)).all() or (row[['Open', 'High', 'Low', 'Close']] <= 0).any() or row['Volume'] < 0:
            continue

        x_left = i * 3        # 1번째 열 (Open)
        x_mid = i * 3 + 1     # 2번째 열 (High-Low 세로선 & MA 점 위치)
        x_right = i * 3 + 2   # 3번째 열 (Close)

        # ---------------- A. 가격 영역 (Prices Area) ----------------
        y_high = int((price_h - 1) - ((row['High'] - p_min) / p_range * (price_h - 1)))
        y_low = int((price_h - 1) - ((row['Low'] - p_min) / p_range * (price_h - 1)))
        y_open = int((price_h - 1) - ((row['Open'] - p_min) / p_range * (price_h - 1)))
        y_close = int((price_h - 1) - ((row['Close'] - p_min) / p_range * (price_h - 1)))

        # High-Low 바 및 Open/Close 틱
        img[y_high : y_low + 1, x_mid] = 255
        img[y_open, x_left] = 255
        img[y_close, x_right] = 255

        # [MA 점 찍기]: 가운데 열(x_mid)에 1픽셀만 255로 표시
        ma_val = ma_series.iloc[i]
        if not np.isnan(ma_val):
            y_ma = int((price_h - 1) - ((ma_val - p_min) / p_range * (price_h - 1)))
            y_ma = np.clip(y_ma, 0, price_h - 1)
            img[y_ma, x_mid] = 255

        # ---------------- B. 거래량 영역 (Volume Area) ----------------
        vol_len = int((row['Volume'] / v_max) * vol_h)
        if vol_len > 0:
            img[height - vol_len : height, x_mid] = 255

    return img



In [5]:
def main():
    start, stop, test_start, test_stop = map(pd.Timestamp, (
        TRAIN_VALID_START, TRAIN_VALID_END, TEST_START, TEST_END))
    if not start <= stop < test_start <= test_stop:
        raise ValueError('학습·검증 기간 이후에 테스트 기간을 설정하세요.')
    history = load_history(EXCEL_DIR)
    calendar = pd.DatetimeIndex(sorted(history['Date'].unique()))
    if not ((calendar >= start) & (calendar <= stop)).any():
        raise ValueError(f'학습 기간 데이터가 없습니다. 보유 기간: {calendar.min()} ~ {calendar.max()}. 설정을 변경하세요.')
    # 기존 산출물과 섞이지 않도록 실행별 폴더를 생성합니다.
    import tempfile
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix='run_', dir=OUTPUT_DIR))
    manifests = {(n, h): [] for n in SPECS for h in HORIZONS}
    for ticker, frame in history.groupby('Ticker', sort=True):
        work = frame.set_index('Date').reindex(calendar)
        for n in SPECS:
            samples = {h: [] for h in HORIZONS}
            for end in range(n - 1, len(work), WINDOW_STEP):
                date = calendar[end]
                pool = 'train_valid' if start <= date <= stop else 'test' if test_start <= date <= test_stop else None
                if pool is None:
                    continue
                window = work.iloc[end - n + 1:end + 1]
                if not (window['Close'].iloc[[0, -1]] > 0).all():
                    continue
                eligible = []
                for h in HORIZONS:
                    if end + h >= len(calendar) or calendar[end + h] > (stop if pool == 'train_valid' else test_stop):
                        continue  # 라벨 기간도 같은 기간 안에 있어야 합니다.
                    ret = future_return(work, end, h)
                    if np.isfinite(ret):
                        eligible.append((h, ret))
                if not eligible:
                    continue
                try:
                    image = generate_ohlcv_image(window, n)
                except ValueError:
                    continue
                for h, ret in eligible:
                    samples[h].append((image, int(ret > 0), ret, date, calendar[end + h], pool))
            for h, rows in samples.items():
                if not rows:
                    continue
                filename = f'{ticker}_input{n}_future{h}.npz'
                np.savez_compressed(run_dir / filename, X=np.stack([r[0] for r in rows]))
                for i, (_, label, ret, date, label_end, pool) in enumerate(rows):
                    manifests[n, h].append(dict(file=filename, row=i, ticker=ticker,
                        date=date, label_end=label_end, y=label, future_return=ret, split=pool))
    for (n, h), records in manifests.items():
        if not records:
            raise ValueError(f'{n}/{h}일 유효 표본이 없습니다. 기간과 RET를 확인하세요. 생성 경로: {run_dir}')
        manifest = pd.DataFrame(records)
        candidates = manifest.index[manifest['split'] == 'train_valid'].to_numpy()
        if len(candidates) < 2:
            raise ValueError('학습·검증 표본이 최소 2개 필요합니다.')
        shuffled = np.random.default_rng(SEED).permutation(candidates)
        cut = max(1, min(len(shuffled) - 1, int(len(shuffled) * 0.7)))
        manifest.loc[shuffled[:cut], 'split'] = 'train'
        manifest.loc[shuffled[cut:], 'split'] = 'validation'
        # 모든 학습 이미지의 모든 픽셀에서 단일 평균/표준편차 계산(ddof=0).
        count, total, squares = 0, 0.0, 0.0
        for file, group in manifest[manifest['split'] == 'train'].groupby('file'):
            with np.load(run_dir / file) as data:
                pixels = data['X'][group['row'].to_numpy()].astype(np.float64) / 255.0
            count += pixels.size
            total += pixels.sum()
            squares += np.square(pixels).sum()
        mean = total / count
        std = float(np.sqrt(max(0, squares / count - mean ** 2)))
        if std == 0:
            raise ValueError('학습 픽셀 표준편차가 0입니다.')
        tag = f'input{n}_future{h}'
        manifest.to_csv(run_dir / f'{tag}.csv', index=False)
        stats = dict(mean=mean, std=std, pixel_scale=255.0, seed=SEED,
                     train_valid_period=[TRAIN_VALID_START, TRAIN_VALID_END],
                     test_period=[TEST_START, TEST_END])
        (run_dir / f'{tag}_normalization.json').write_text(json.dumps(stats, indent=2))
        print(tag, manifest['split'].value_counts().to_dict())
    print(f'완료: {run_dir}')
    return run_dir


def load_normalized_shard(run_dir, input_days, horizon, filename, split):
    """지정 shard/split의 CNN 입력(float32), 라벨, 메타데이터를 반환합니다."""
    run_dir = Path(run_dir)
    tag = f'input{input_days}_future{horizon}'
    manifest = pd.read_csv(run_dir / f'{tag}.csv', dtype={'ticker': str})
    rows = manifest[(manifest['file'] == filename) & (manifest['split'] == split)]
    stats = json.loads((run_dir / f'{tag}_normalization.json').read_text())
    with np.load(run_dir / filename) as data:
        x = data['X'][rows['row'].to_numpy()].astype(np.float32) / stats['pixel_scale']
    x = (x - np.float32(stats['mean'])) / np.float32(stats['std'])
    return x[:, None, :, :], rows['y'].to_numpy(np.int64), rows


In [6]:
run_dir = main()

input5_future20 {'train': 1211249, 'validation': 519108, 'test': 429688}
input5_future60 {'train': 1171902, 'validation': 502245, 'test': 388765}
input20_future20 {'train': 1196358, 'validation': 512726, 'test': 427950}
input20_future60 {'train': 1157442, 'validation': 496047, 'test': 387599}
input60_future20 {'train': 1157442, 'validation': 496047, 'test': 424490}
input60_future60 {'train': 1119363, 'validation': 479727, 'test': 384598}
완료: /Users/claremoon/vision-retail/processed_data/run_ep087wab
